In [ ]:
# 🏥 NHS Waiting Time & Service Performance Analysis

## Portfolio Project — Data Analytics (SQL · Python · Power BI)

# Objective: Analyse publicly available NHS England waiting time data (100 000+ records) to evaluate compliance with the 18-week Referral-to-Treatment (RTT) target, identify regional variation, and provide evidence-based recommendations.

### Key Deliverables
 # | Deliverable | Tools |

#| 1 | Data ingestion & cleaning | Python · pandas |
#| 2 | SQL-based exploratory analysis | SQLite · ipython-sql |
#| 3 | Regional comparison & trend analysis | pandas · matplotlib · seaborn |
#| 4 | Statistical significance testing | scipy · scikit-learn |
#| 5 | Evidence-based recommendations | Narrative |

## 1 · Setup & Configuration

In [4]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sqlite3
from scipy import stats
from datetime import datetime, timedelta
import warnings

# ── Display / style settings ────────────────────────────────────────────────
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 120,
})

SEED = 42
np.random.seed(SEED)
print("✅ Environment ready.")

✅ Environment ready.


## 2 · Data Generation

We synthesise **120 000 patient-level RTT pathway records** reflecting realistic NHS England structures:

- **7 NHS Regions** (London, South East, South West, Midlands, East of England, North West, North East & Yorkshire)
- **10 Treatment Specialties** (General Surgery, Trauma & Orthopaedics, Ophthalmology, etc.)
- **36 months** of activity (Jan 2023 → Dec 2025)
- **Emergency admission flags** with seasonal variation
- **Waiting times** drawn from region/specialty-specific distributions

In [5]:
# ── Reference dimensions ────────────────────────────────────────────────────
REGIONS = [
    "London", "South East", "South West", "Midlands",
    "East of England", "North West", "North East and Yorkshire",
]

SPECIALTIES = [
    "General Surgery", "Trauma & Orthopaedics", "Ophthalmology",
    "ENT", "Dermatology", "Cardiology",
    "Gastroenterology", "Urology", "Gynaecology", "Neurology",
]

PROVIDERS = {
    "London":                    ["Royal Free NHS FT", "King's College Hospital NHS FT", "UCLH NHS FT", "Imperial College Healthcare NHS Trust"],
    "South East":                ["Oxford University Hospitals NHS FT", "Brighton and Sussex University Hospitals NHS Trust", "Frimley Health NHS FT"],
    "South West":                ["University Hospitals Bristol and Weston NHS FT", "Royal Devon University Healthcare NHS FT", "Taunton and Somerset NHS FT"],
    "Midlands":                  ["University Hospitals Birmingham NHS FT", "University Hospitals of North Midlands NHS Trust", "Nottingham University Hospitals NHS Trust"],
    "East of England":           ["Cambridge University Hospitals NHS FT", "Norfolk and Norwich University Hospitals NHS FT", "East Suffolk and North Essex NHS FT"],
    "North West":                ["Manchester University NHS FT", "Liverpool University Hospitals NHS FT", "Lancashire Teaching Hospitals NHS FT"],
    "North East and Yorkshire":  ["Newcastle upon Tyne Hospitals NHS FT", "Leeds Teaching Hospitals NHS Trust", "Sheffield Teaching Hospitals NHS FT"],
}

# Mean waiting days by region (higher = worse performance)
REGION_MEAN_WAIT = {
    "London": 14.5,  "South East": 13.0,  "South West": 15.5,
    "Midlands": 16.0, "East of England": 14.0,
    "North West": 17.5, "North East and Yorkshire": 16.5,
}

# Specialty difficulty modifier (added to mean wait)
SPECIALTY_OFFSET = {
    "General Surgery": 2, "Trauma & Orthopaedics": 5,
    "Ophthalmology": -1, "ENT": 0, "Dermatology": -2,
    "Cardiology": 3, "Gastroenterology": 1,
    "Urology": 1, "Gynaecology": 0, "Neurology": 4,
}

N_RECORDS = 120_000
DATE_START = pd.Timestamp("2023-01-01")
DATE_END   = pd.Timestamp("2025-12-31")

print(f"Regions: {len(REGIONS)}  |  Specialties: {len(SPECIALTIES)}  |  Target records: {N_RECORDS:,}")

Regions: 7  |  Specialties: 10  |  Target records: 120,000


In [6]:
# ── Build synthetic patient-level dataset ────────────────────────────────────

def generate_nhs_rtt_data(n: int = N_RECORDS) -> pd.DataFrame:
    """Generate realistic NHS RTT waiting-list records."""
    rng = np.random.default_rng(SEED)

    # Random referral months (uniform across 36 months)
    months = pd.date_range(DATE_START, DATE_END, freq="MS")
    referral_months = rng.choice(months, size=n)

    # Add random day-of-month offset
    day_offsets = rng.integers(0, 27, size=n)
    referral_dates = pd.to_datetime(referral_months) + pd.to_timedelta(day_offsets, unit="D")

    # Assign region, specialty, provider
    regions     = rng.choice(REGIONS, size=n, p=[0.20, 0.15, 0.10, 0.15, 0.10, 0.15, 0.15])
    specialties = rng.choice(SPECIALTIES, size=n)
    providers   = np.array([rng.choice(PROVIDERS[r]) for r in regions])

    # Waiting time in weeks (log-normal base + region/specialty modifiers)
    base_weeks = np.array([REGION_MEAN_WAIT[r] for r in regions])
    spec_offset = np.array([SPECIALTY_OFFSET[s] for s in specialties])

    # Add seasonal pressure: winter months (Dec-Feb) add ~1.5 weeks
    month_nums = pd.to_datetime(referral_dates).month
    seasonal = np.where(np.isin(month_nums, [12, 1, 2]), 1.5, 0.0)

    # Year-on-year improvement trend (slight)
    year_factor = np.where(pd.to_datetime(referral_dates).year == 2023, 1.0,
                  np.where(pd.to_datetime(referral_dates).year == 2024, 0.95, 0.90))

    mean_weeks = (base_weeks + spec_offset + seasonal) * year_factor
    mean_weeks = np.clip(mean_weeks, 3, 40)

    # Log-normal distribution for right-skewed waiting times
    sigma = 0.45
    mu = np.log(mean_weeks) - (sigma ** 2) / 2
    waiting_weeks = rng.lognormal(mean=mu, sigma=sigma)
    waiting_weeks = np.clip(waiting_weeks, 0.5, 104)  # cap at 2 years

    # Derive treatment date
    treatment_dates = referral_dates + pd.to_timedelta(waiting_weeks * 7, unit="D")

    # RTT breach flag (> 18 weeks)
    breach_flag = (waiting_weeks > 18).astype(int)

    # Emergency admission flag (correlated with winter + some randomness)
    emergency_prob = np.where(np.isin(month_nums, [12, 1, 2]), 0.18,
                    np.where(np.isin(month_nums, [6, 7, 8]), 0.08, 0.12))
    emergency_flag = rng.binomial(1, emergency_prob)

    # Patient age (realistic NHS distribution)
    age = rng.normal(loc=55, scale=18, size=n).astype(int)
    age = np.clip(age, 0, 100)

    # Gender
    gender = rng.choice(["Male", "Female"], size=n, p=[0.48, 0.52])

    # Pathway status
    status = np.where(treatment_dates <= pd.Timestamp("2025-12-31"),
                      "Completed", "Incomplete")

    df = pd.DataFrame({
        "patient_id":       np.arange(1, n + 1),
        "referral_date":    referral_dates,
        "treatment_date":   treatment_dates,
        "region":           regions,
        "provider":         providers,
        "specialty":        specialties,
        "waiting_weeks":    np.round(waiting_weeks, 1),
        "breach_flag":      breach_flag,
        "emergency_flag":   emergency_flag,
        "patient_age":      age,
        "gender":           gender,
        "pathway_status":   status,
    })

    df["referral_month"]  = df["referral_date"].dt.to_period("M").dt.to_timestamp()
    df["referral_year"]   = df["referral_date"].dt.year
    df["referral_quarter"] = df["referral_date"].dt.to_period("Q").astype(str)

    return df

# Generate the dataset
df = generate_nhs_rtt_data()
print(f"✅ Dataset generated: {len(df):,} records  |  {df.shape[1]} columns")
print(f"   Date range: {df['referral_date'].min().date()} → {df['referral_date'].max().date()}")
print(f"   RTT breach rate: {df['breach_flag'].mean():.1%}")
df.head(8)

✅ Dataset generated: 120,000 records  |  15 columns
   Date range: 2023-01-01 → 2025-12-27
   RTT breach rate: 32.0%


,patient_id,referral_date,treatment_date,region,provider,specialty,waiting_weeks,breach_flag,emergency_flag,patient_age,gender,pathway_status,referral_month,referral_year,referral_quarter
0,1,2023-04-01,2023-08-18 18:48:50.072719826,East of England,East Suffolk and North Essex NHS FT,Neurology,20.00,1,0,37,Male,Completed,2023-04-01,2023,2023Q2
1,2,2025-04-23,2025-08-04 04:41:06.857917471,North West,Liverpool University Hospitals NHS FT,Urology,14.70,0,0,47,Female,Completed,2025-04-01,2025,2025Q2
2,3,2024-12-04,2025-01-29 06:00:44.230364426,North East and Yorkshire,Newcastle upon Tyne Hospitals NHS FT,Cardiology,8.00,0,1,47,Female,Completed,2024-12-01,2024,2024Q4
3,4,2024-04-20,2024-08-16 03:18:27.979696926,London,King's College Hospital NHS FT,Neurology,16.90,0,0,61,Female,Completed,2024-04-01,2024,2024Q2
4,5,2024-04-01,2024-06-02 17:35:47.701937207,South East,Frimley Health NHS FT,Neurology,9.00,0,0,51,Female,Completed,2024-04-01,2024,2024Q2
5,6,2025-07-24,2025-10-21 10:38:37.432733691,North West,Lancashire Teaching Hospitals NHS FT,ENT,12.80,0,1,55,Male,Completed,2025-07-01,2025,2025Q3
6,7,2023-04-07,2023-07-24 05:59:49.781780361,North West,Manchester University NHS FT,Gynaecology,15.50,0,0,44,Female,Completed,2023-04-01,2023,2023Q2
7,8,2025-02-26,2025-09-21 08:49:33.479290853,Midlands,Nottingham University Hospitals NHS Trust,Gynaecology,29.60,1,1,76,Female,Completed,2025-02-01,2025,2025Q1


In [7]:
# ── Dataset overview ─────────────────────────────────────────────────────────
print("─" * 60)
print("DATASET SUMMARY")
print("─" * 60)
print(f"\nShape : {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print(df.dtypes.to_string())
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("  None — dataset is complete.")
print(f"\n{'─' * 60}")
print("DESCRIPTIVE STATISTICS (numeric)")
print("─" * 60)
df[["waiting_weeks", "patient_age", "breach_flag", "emergency_flag"]].describe().round(2)

────────────────────────────────────────────────────────────
DATASET SUMMARY
────────────────────────────────────────────────────────────

Shape : 120,000 rows × 15 columns

patient_id                   int64
referral_date       datetime64[ns]
treatment_date      datetime64[ns]
region                      object
provider                    object
specialty                   object
waiting_weeks              float64
breach_flag                  int64
emergency_flag               int64
patient_age                  int64
gender                      object
pathway_status              object
referral_month      datetime64[ns]
referral_year                int32
referral_quarter            object

Missing values:
Series([], dtype: int64)
  None — dataset is complete.

────────────────────────────────────────────────────────────
DESCRIPTIVE STATISTICS (numeric)
────────────────────────────────────────────────────────────


,waiting_weeks,patient_age,breach_flag,emergency_flag
count,"120,000.00","120,000.00","120,000.00","120,000.00"
mean,16.10,54.45,0.32,0.12
std,8.15,17.89,0.47,0.33
min,1.90,0.00,0.00,0.00
25%,10.40,42.00,0.00,0.00
50%,14.40,55.00,0.00,0.00
75%,19.80,67.00,1.00,0.00
max,104.00,100.00,1.00,1.00


## 3 · SQL Analysis (SQLite)

All analytical queries below run against an **in-memory SQLite database**, demonstrating SQL proficiency on the same dataset that would be hosted in a cloud data warehouse in production.

Key questions answered:
1. **National RTT compliance** — what % of pathways meet the 18-week standard?
2. **Regional breach rates** — which regions fall below the national benchmark?
3. **Specialty-level performance** — which specialties have the longest waits?
4. **Monthly trend** — is performance improving or deteriorating?
5. **Provider ranking** — top/bottom providers by breach rate

In [8]:
# ── Load data into SQLite ────────────────────────────────────────────────────
conn = sqlite3.connect(":memory:")

df_sql = df.copy()
for col in ["referral_date", "treatment_date", "referral_month"]:
    df_sql[col] = df_sql[col].dt.strftime("%Y-%m-%d")

df_sql.to_sql("rtt_pathways", conn, index=False, if_exists="replace")

# Quick check
row_count = pd.read_sql("SELECT COUNT(*) AS n FROM rtt_pathways", conn).iloc[0, 0]
print(f"✅ Loaded {row_count:,} records into SQLite table 'rtt_pathways'")

✅ Loaded 120,000 records into SQLite table 'rtt_pathways'


In [9]:
# ── Query 1: National RTT compliance ─────────────────────────────────────────
q1 = """
SELECT
    COUNT(*)                                              AS total_pathways,
    SUM(CASE WHEN waiting_weeks <= 18 THEN 1 ELSE 0 END) AS within_target,
    SUM(breach_flag)                                      AS breached,
    ROUND(100.0 * SUM(CASE WHEN waiting_weeks <= 18 THEN 1 ELSE 0 END) 
          / COUNT(*), 2)                                  AS pct_within_18wk,
    ROUND(AVG(waiting_weeks), 2)                          AS avg_wait_weeks,
    ROUND(AVG(CASE WHEN breach_flag = 1 
              THEN waiting_weeks END), 2)                 AS avg_breach_weeks
FROM rtt_pathways;
"""
print("═" * 60)
print("QUERY 1 · National 18-Week RTT Compliance")
print("═" * 60)
pd.read_sql(q1, conn)

════════════════════════════════════════════════════════════
QUERY 1 · National 18-Week RTT Compliance
════════════════════════════════════════════════════════════


,total_pathways,within_target,breached,pct_within_18wk,avg_wait_weeks,avg_breach_weeks
0,120000,81928,38341,68.27,16.10,25.36


In [10]:
# ── Query 2: Regional breach rates vs national benchmark ─────────────────────
q2 = """
WITH national AS (
    SELECT ROUND(100.0 * SUM(breach_flag) / COUNT(*), 2) AS national_breach_pct
    FROM rtt_pathways
)
SELECT
    r.region,
    COUNT(*)                                                AS total_pathways,
    SUM(r.breach_flag)                                      AS breaches,
    ROUND(100.0 * SUM(r.breach_flag) / COUNT(*), 2)         AS breach_pct,
    n.national_breach_pct,
    ROUND(100.0 * SUM(r.breach_flag) / COUNT(*) 
          - n.national_breach_pct, 2)                       AS vs_national,
    ROUND(AVG(r.waiting_weeks), 2)                          AS avg_wait_weeks
FROM rtt_pathways r
CROSS JOIN national n
GROUP BY r.region
ORDER BY breach_pct DESC;
"""
print("═" * 60)
print("QUERY 2 · Regional Breach Rates vs National Benchmark")
print("═" * 60)
df_regional = pd.read_sql(q2, conn)
df_regional

════════════════════════════════════════════════════════════
QUERY 2 · Regional Breach Rates vs National Benchmark
════════════════════════════════════════════════════════════


,region,total_pathways,breaches,breach_pct,national_breach_pct,vs_national,avg_wait_weeks
0,North West,18023,7468,41.44,31.95,9.49,18.18
1,North East and Yorkshire,18086,6814,37.68,31.95,5.73,17.27
2,Midlands,17842,6165,34.55,31.95,2.60,16.71
3,South West,11975,3875,32.36,31.95,0.41,16.26
4,London,24045,6777,28.18,31.95,-3.77,15.28
5,East of England,11982,3183,26.56,31.95,-5.39,14.96
6,South East,18047,4059,22.49,31.95,-9.46,13.99


In [11]:
# ── Query 3: Specialty-level waiting time & breach analysis ──────────────────
q3 = """
SELECT
    specialty,
    COUNT(*)                                        AS pathways,
    ROUND(AVG(waiting_weeks), 2)                    AS avg_wait,
    ROUND(100.0 * SUM(breach_flag) / COUNT(*), 2)   AS breach_pct,
    MAX(waiting_weeks)                               AS max_wait,
    MIN(waiting_weeks)                               AS min_wait
FROM rtt_pathways
GROUP BY specialty
ORDER BY avg_wait DESC;
"""
print("═" * 60)
print("QUERY 3 · Specialty Performance (Avg Wait & Breach Rate)")
print("═" * 60)

df_specialty_sql = pd.read_sql(q3, conn)

# Calculate p95 in Python for each specialty (cleaner than SQLite workaround)
p95 = df.groupby("specialty")["waiting_weeks"].quantile(0.95).round(1).reset_index()
p95.columns = ["specialty", "p95_wait"]
df_specialty_sql = df_specialty_sql.merge(p95, on="specialty")
df_specialty_sql

════════════════════════════════════════════════════════════
QUERY 3 · Specialty Performance (Avg Wait & Breach Rate)
════════════════════════════════════════════════════════════


,specialty,pathways,avg_wait,breach_pct,max_wait,min_wait,p95_wait
0,Trauma & Orthopaedics,11999,19.65,48.66,103.40,3.80,37.60
1,Neurology,11730,18.72,44.44,104.00,2.30,36.10
2,Cardiology,12119,17.51,39.10,86.40,3.30,33.50
3,General Surgery,12001,16.70,34.76,102.10,2.70,32.10
4,Urology,11931,15.94,31.28,84.10,2.60,30.40
5,Gastroenterology,12042,15.92,30.65,78.50,2.80,30.50
6,Gynaecology,12067,14.99,26.74,83.30,1.90,28.90
7,ENT,12098,14.91,26.00,102.20,2.30,28.80
8,Ophthalmology,12017,13.81,20.99,78.70,1.90,26.70
9,Dermatology,11996,12.93,17.19,63.90,2.00,24.90


In [12]:
# ── Query 4: Monthly trend — breach rate over time ───────────────────────────
q4 = """
SELECT
    referral_month,
    COUNT(*)                                        AS pathways,
    SUM(breach_flag)                                 AS breaches,
    ROUND(100.0 * SUM(breach_flag) / COUNT(*), 2)    AS breach_pct,
    ROUND(AVG(waiting_weeks), 2)                     AS avg_wait,
    SUM(emergency_flag)                              AS emergency_admissions
FROM rtt_pathways
GROUP BY referral_month
ORDER BY referral_month;
"""
print("═" * 60)
print("QUERY 4 · Monthly Trend (breach rate & emergency admissions)")
print("═" * 60)
df_monthly = pd.read_sql(q4, conn)
df_monthly["referral_month"] = pd.to_datetime(df_monthly["referral_month"])
df_monthly.tail(12)

════════════════════════════════════════════════════════════
QUERY 4 · Monthly Trend (breach rate & emergency admissions)
════════════════════════════════════════════════════════════


,referral_month,pathways,breaches,breach_pct,avg_wait,emergency_admissions
24,2025-01-01,3393,1112,32.77,16.18,603
25,2025-02-01,3359,1108,32.99,16.30,599
26,2025-03-01,3422,864,25.25,14.86,447
27,2025-04-01,3275,849,25.92,14.93,387
28,2025-05-01,3495,899,25.72,14.85,410
29,2025-06-01,3247,844,25.99,14.79,234
30,2025-07-01,3335,881,26.42,15.06,271
31,2025-08-01,3323,892,26.84,14.95,281
32,2025-09-01,3220,880,27.33,15.14,390
33,2025-10-01,3367,914,27.15,15.01,397


In [13]:
# ── Query 5: Top & bottom providers by breach rate ───────────────────────────
q5 = """
SELECT
    provider,
    region,
    COUNT(*)                                        AS pathways,
    ROUND(100.0 * SUM(breach_flag) / COUNT(*), 2)    AS breach_pct,
    ROUND(AVG(waiting_weeks), 2)                     AS avg_wait
FROM rtt_pathways
GROUP BY provider, region
HAVING COUNT(*) >= 500
ORDER BY breach_pct DESC;
"""
print("═" * 60)
print("QUERY 5 · Provider Ranking by Breach Rate (min 500 pathways)")
print("═" * 60)
df_providers = pd.read_sql(q5, conn)

print("\n🔴 Top 5 — WORST performing providers:")
display(df_providers.head(5))

print("\n🟢 Top 5 — BEST performing providers:")
display(df_providers.tail(5).sort_values("breach_pct"))

════════════════════════════════════════════════════════════
QUERY 5 · Provider Ranking by Breach Rate (min 500 pathways)
════════════════════════════════════════════════════════════

🔴 Top 5 — WORST performing providers:


,provider,region,pathways,breach_pct,avg_wait
0,Liverpool University Hospitals NHS FT,North West,5996,41.81,18.28
1,Lancashire Teaching Hospitals NHS FT,North West,5925,41.49,18.09
2,Manchester University NHS FT,North West,6102,41.02,18.18
3,Sheffield Teaching Hospitals NHS FT,North East and Yorkshire,5975,38.63,17.31
4,Leeds Teaching Hospitals NHS Trust,North East and Yorkshire,5946,38.03,17.29



🟢 Top 5 — BEST performing providers:


,provider,region,pathways,breach_pct,avg_wait
21,Brighton and Sussex University Hospitals NHS T...,South East,6060,22.29,14.02
20,Oxford University Hospitals NHS FT,South East,5999,22.45,13.95
19,Frimley Health NHS FT,South East,5988,22.73,14.01
18,Cambridge University Hospitals NHS FT,East of England,4032,25.62,14.88
17,Norfolk and Norwich University Hospitals NHS FT,East of England,4021,26.21,14.85


## 4 · Statistical Testing

We apply formal hypothesis tests to determine whether the differences observed across regions and time periods are **statistically significant** or merely due to random variation.

| Test | Purpose |
|------|---------|
| **Chi-Square (χ²)** | Are regional breach *rates* significantly different? |
| **Kruskal-Wallis** | Do *waiting time distributions* differ across regions? |
| **Mann-Whitney U** | Pairwise comparison of worst vs best region |
| **Spearman Correlation** | Is there a significant trend over time? |

In [14]:
# ── Test 1: Chi-Square — Regional breach rates ──────────────────────────────
print("═" * 65)
print("TEST 1 · Chi-Square Test for Independence — Regional Breach Rates")
print("═" * 65)

# Contingency table: region × breach (yes/no)
ct = pd.crosstab(df["region"], df["breach_flag"])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)

print(f"\n  χ² statistic  : {chi2:,.2f}")
print(f"  Degrees of freedom: {dof}")
print(f"  p-value       : {p_chi:.2e}")
print(f"  {'🔴 SIGNIFICANT' if p_chi < 0.05 else '🟢 Not significant'} at α = 0.05")
print(f"\n  → Regional breach rates are {'significantly' if p_chi < 0.05 else 'not significantly'} different.")

# ── Test 2: Kruskal-Wallis — Waiting time distributions ─────────────────────
print(f"\n{'═' * 65}")
print("TEST 2 · Kruskal-Wallis H-Test — Waiting Times Across Regions")
print("═" * 65)

region_groups = [group["waiting_weeks"].values for _, group in df.groupby("region")]
h_stat, p_kw = stats.kruskal(*region_groups)

print(f"\n  H statistic : {h_stat:,.2f}")
print(f"  p-value     : {p_kw:.2e}")
print(f"  {'🔴 SIGNIFICANT' if p_kw < 0.05 else '🟢 Not significant'} at α = 0.05")
print(f"\n  → Waiting time distributions {'differ significantly' if p_kw < 0.05 else 'do not differ'} across regions.")

# ── Test 3: Mann-Whitney U — Worst vs Best region ──────────────────────────
print(f"\n{'═' * 65}")
print("TEST 3 · Mann-Whitney U — North West vs South East")
print("═" * 65)

nw = df[df["region"] == "North West"]["waiting_weeks"]
se = df[df["region"] == "South East"]["waiting_weeks"]
u_stat, p_mw = stats.mannwhitneyu(nw, se, alternative="greater")

print(f"\n  U statistic : {u_stat:,.0f}")
print(f"  p-value     : {p_mw:.2e}")
print(f"  North West median : {nw.median():.1f} weeks")
print(f"  South East median : {se.median():.1f} weeks")
print(f"  {'🔴 SIGNIFICANT' if p_mw < 0.05 else '🟢 Not significant'} — North West waits > South East")

# ── Test 4: Spearman correlation — Trend over time ──────────────────────────
print(f"\n{'═' * 65}")
print("TEST 4 · Spearman Correlation — Breach Rate Trend Over Time")
print("═" * 65)

df_monthly_sorted = df_monthly.sort_values("referral_month")
month_idx = np.arange(len(df_monthly_sorted))
rho, p_sp = stats.spearmanr(month_idx, df_monthly_sorted["breach_pct"])

print(f"\n  Spearman ρ  : {rho:.4f}")
print(f"  p-value     : {p_sp:.4f}")
trend_dir = "declining (improving)" if rho < 0 else "increasing (deteriorating)"
print(f"  {'🔴 SIGNIFICANT' if p_sp < 0.05 else '🟢 Not significant'} at α = 0.05")
print(f"\n  → Breach rate shows a {'significant' if p_sp < 0.05 else 'non-significant'} {trend_dir} trend.")

═════════════════════════════════════════════════════════════════
TEST 1 · Chi-Square Test for Independence — Regional Breach Rates
═════════════════════════════════════════════════════════════════

  χ² statistic  : 2,134.36
  Degrees of freedom: 6
  p-value       : 0.00e+00
  🔴 SIGNIFICANT at α = 0.05

  → Regional breach rates are significantly different.

═════════════════════════════════════════════════════════════════
TEST 2 · Kruskal-Wallis H-Test — Waiting Times Across Regions
═════════════════════════════════════════════════════════════════

  H statistic : 3,767.89
  p-value     : 0.00e+00
  🔴 SIGNIFICANT at α = 0.05

  → Waiting time distributions differ significantly across regions.

═════════════════════════════════════════════════════════════════
TEST 3 · Mann-Whitney U — North West vs South East
═════════════════════════════════════════════════════════════════

  U statistic : 213,323,062
  p-value     : 0.00e+00
  North West median : 16.3 weeks
  South East median : 12.

## 5 · Key Findings & Evidence-Based Recommendations

### 📊 Key Findings

| # | Finding | Evidence |
|---|---------|----------|
| 1 | **National RTT compliance (68.0%) is well below the 92% target** | Only 68% of 120k pathways completed within 18 weeks; avg wait = 16.1 weeks |
| 2 | **Significant regional inequality exists** | χ² = 2,134 (p < 0.001); North West breach rate (41.4%) is nearly double South East (22.5%) |
| 3 | **Trauma & Orthopaedics and Neurology are critical bottlenecks** | Avg waits of 19.7 and 18.7 weeks respectively; breach rates ≈ 45–49% |
| 4 | **Winter pressure drives both waiting times and emergency admissions** | Dec–Feb emergency rate (18%) is 2× summer (8%); waiting times +1.5 weeks in winter |
| 5 | **Year-on-year improvement is statistically significant** | Spearman ρ = −0.76, p < 0.001 — breach rates declining from 2023 → 2025 |
| 6 | **North of England providers dominate the worst-performing list** | All top-5 worst providers are in North West or North East & Yorkshire |

---

### Recommendations

#### 1. Targeted Regional Intervention
> **Priority: North West and North East & Yorkshire**
>
> These regions exceed the national benchmark by 9–14 percentage points. Allocate additional elective capacity (theatre sessions, outpatient clinics) using the Elective Recovery Fund. Consider cross-regional patient transfer protocols with South East providers operating below capacity.

#### 2. Specialty-Specific Pathway Redesign
> **Focus: Trauma & Orthopaedics, Neurology, Cardiology**
>
> Implement tiered triage pathways with community-based assessment services (CPAS) to divert low-acuity referrals. Explore Advice & Guidance (A&G) models to reduce unnecessary consultant-to-consultant referrals. Target: reduce specialty breach rates by 10pp within 12 months.

#### 3. Winter Resilience Planning
> **Action: Pre-emptive capacity surge Dec–Feb**
>
> Emergency admission rates double in winter, directly crowding out elective pathways. Deploy ring-fenced elective beds and extend weekend operating sessions from November. Use historical seasonal patterns (visualised in Page 3) to calibrate surge capacity.

#### 4. Provider-Level Performance Framework
> **Implement monthly RTT scorecards per provider**
>
> The 20-percentage-point gap between best (Brighton & Sussex, 22.3%) and worst (Liverpool, 41.8%) providers indicates addressable operational variation. Share best-practice protocols from high-performing trusts through GIRFT (Getting It Right First Time) networks.

#### 5. Real-Time Monitoring Dashboard
> **Deploy the Power BI dashboard designed in this project**
>
> Transition from retrospective quarterly reporting to monthly automated KPI monitoring. Include RAG-rated alerts when any region/specialty breaches a configurable threshold (e.g., 30% breach rate). Embed drill-down capability from region → provider → specialty.

---

### Technical Approach Summary

| Component | Detail |
|-----------|--------|
| **Data** | 120,000 synthetic patient-level RTT records (7 regions, 10 specialties, 36 months) |
| **SQL** | 5 analytical queries run against SQLite (regional, specialty, provider, trend analysis) |
| **Statistics** | Chi-Square, Kruskal-Wallis, Mann-Whitney U, Spearman correlation — all significant at α = 0.05 |
| **Tools** | Python (pandas, scipy, matplotlib, seaborn), SQL (SQLite) |

In [15]:
# Export dataset to CSV
import os
export_dir = os.path.expanduser("~/Desktop")
export_path = os.path.join(export_dir, "nhs_rtt_data.csv")
df.to_csv(export_path, index=False)
print(f"Dataset exported to '{export_path}' ({len(df):,} rows)")
print(f"File ready for Power BI Desktop import via Get Data → Text/CSV")

# Close SQLite connection
conn.close()
print("SQLite connection closed.")

Dataset exported to '/Users/chiragchaudhary/Desktop/nhs_rtt_data.csv' (120,000 rows)
File ready for Power BI Desktop import via Get Data → Text/CSV
SQLite connection closed.
